In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/kickstarter_us_filtered.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (38504, 17)


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
3,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00
4,316217,Dinner Party - a short film about friendship.....,http://www.kickstarter.com/projects/249354515/...,Film & Video,Short Film,"Portland, OR",successful,3500.0,3582.0,1.023331,39,"Wed, 22 Jun 2011 13:33:00 -0000",7,"$5,$25,$50,$100,$250,$500,$1,000",8,0,21.43


In [2]:
# Turn successful (1) and failed (0) to integers for modeling
df["success"] = (df["status"] == "successful").astype(int)
print(df["success"].value_counts())
print("\nSuccess rate:", round(df["success"].mean() * 100, 2), "%")

success
1    21075
0    17429
Name: count, dtype: int64

Success rate: 54.73 %


In [3]:
# Extract state from location
df["state"] = (
    df["location"]
    .str.rsplit(",", n=1)
    .str[-1]
    .str.strip()
)
print("Unique states:", df["state"].nunique())
df["state"].value_counts().head()

Unique states: 51


state
CA    8220
NY    6639
TX    1847
IL    1720
FL    1410
Name: count, dtype: int64

In [4]:
# Define initial feature set
numerical_features = ["goal", "duration", "levels"]
categorical_features = ["category", "subcategory", "state"]
selected_features = numerical_features + categorical_features

In [5]:
# Information used to make prediction
X = df[selected_features].copy()
# Answer we're trying to predict
y = df["success"].copy()

In [6]:
# Validate the modeling data
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing Values:")
print(X.isnull().sum())

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (38504, 6)
Target shape: (38504,)

Missing Values:
goal           0
duration       0
levels         0
category       0
subcategory    0
state          0
dtype: int64

Target distribution:
success
1    21075
0    17429
Name: count, dtype: int64


In [7]:
# Display the first few rows of the feature matrix
X.head()

,goal,duration,levels,category,subcategory,state
0,10500.0,30.00,7,Film & Video,Short Film,MO
1,4000.0,47.18,5,Games,Board & Card Games,NJ
2,6000.0,32.22,13,Film & Video,Documentary,CA
3,3500.0,30.00,6,Fashion,Fashion,MI
4,3500.0,21.43,7,Film & Video,Short Film,OR


In [8]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [9]:
# Validate the training and testing sets
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining success rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting success rate:")
print(y_test.value_counts(normalize=True))

Training rows: 30803
Testing rows: 7701

Training success rate:
success
1    0.547349
0    0.452651
Name: proportion, dtype: float64

Testing success rate:
success
1    0.547332
0    0.452668
Name: proportion, dtype: float64


In [10]:
# Define the preprocessing steps for numerical and categorical features
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [11]:
print(X_train.shape, X_test.shape)

(30803, 6) (7701, 6)


In [12]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss",
    ))
])

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['goal','duration','levels','category','subcategory','state']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining co

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "confusion_matrix": confusion_matrix(y_test, y_pred)
}

print("Accuracy:", round(metrics["accuracy"], 3))
print("Precision:", round(metrics["precision"], 3))
print("Recall:", round(metrics["recall"], 3))
print("F1-Score:", round(metrics["f1_score"], 3))
print("ROC AUC:", round(metrics["roc_auc"], 3))
print("Confusion Matrix:\n", metrics["confusion_matrix"])

Accuracy: 0.702
Precision: 0.706
Recall: 0.78
F1-Score: 0.741
ROC AUC: 0.762
Confusion Matrix:
 [[2117 1369]
 [ 929 3286]]


In [14]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 4, 6]
}

grid_search = GridSearchCV(pipeline, param_grid, scoring="roc_auc", cv=5, n_jobs=-1)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

tuned_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "confusion_matrix": confusion_matrix(y_test, y_pred)
}

print(f"Accuracy:  {tuned_metrics['accuracy']:.3f}")
print(f"Precision: {tuned_metrics['precision']:.3f}")
print(f"Recall:    {tuned_metrics['recall']:.3f}")
print(f"F1-Score:  {tuned_metrics['f1_score']:.3f}")
print(f"ROC AUC:   {tuned_metrics['roc_auc']:.3f}")
print("Confusion Matrix:\n", tuned_metrics["confusion_matrix"])

{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 400}
0.7623564675836809
Accuracy:  0.703
Precision: 0.702
Recall:    0.795
F1-Score:  0.746
ROC AUC:   0.767
Confusion Matrix:
 [[2067 1419]
 [ 866 3349]]


In [15]:
param_grid_2 = {
    "model__n_estimators": [400, 600, 800],
    "model__learning_rate": [0.1, 0.15, 0.2],
    "model__max_depth": [2, 3, 4]
}

grid_search_2 = GridSearchCV(pipeline, param_grid_2, scoring="roc_auc", cv=5, n_jobs=-1)
grid_search_2.fit(X_train, y_train)

print(grid_search_2.best_params_)
print(grid_search_2.best_score_)

best_model_2 = grid_search_2.best_estimator_
y_pred_2 = best_model_2.predict(X_test)
y_prob_2 = best_model_2.predict_proba(X_test)[:, 1]

tuned_metrics_2 = {
    "accuracy": accuracy_score(y_test, y_pred_2),
    "precision": precision_score(y_test, y_pred_2),
    "recall": recall_score(y_test, y_pred_2),
    "f1_score": f1_score(y_test, y_pred_2),
    "roc_auc": roc_auc_score(y_test, y_prob_2),
    "confusion_matrix": confusion_matrix(y_test, y_pred_2)
}

print(f"Accuracy:  {tuned_metrics_2['accuracy']:.3f}")
print(f"Precision: {tuned_metrics_2['precision']:.3f}")
print(f"Recall:    {tuned_metrics_2['recall']:.3f}")
print(f"F1-Score:  {tuned_metrics_2['f1_score']:.3f}")
print(f"ROC AUC:   {tuned_metrics_2['roc_auc']:.3f}")
print("Confusion Matrix:\n", tuned_metrics_2["confusion_matrix"])

{'model__learning_rate': 0.2, 'model__max_depth': 2, 'model__n_estimators': 400}
0.763823781419331
Accuracy:  0.703
Precision: 0.703
Recall:    0.793
F1-Score:  0.745
ROC AUC:   0.767
Confusion Matrix:
 [[2073 1413]
 [ 874 3341]]


## Final XGBoost model selection

| Model | CV ROC-AUC (train) | Test Accuracy | Test Recall | Test F1 | Test ROC-AUC |
|---|---|---|---|---|---|
| Default | n/a | 0.702 | 0.780 | 0.741 | 0.762 |
| Tuned (round 1) | 0.7624 | 0.703 | 0.795 | 0.746 | 0.767 |
| Tuned (round 2) | 0.7638 | 0.703 | 0.793 | 0.745 | 0.767 |

**Selected model: Tuned (round 1)** — `learning_rate=0.1, max_depth=3, n_estimators=400` (XGBoost 3.4.1).

The two tuned models are effectively tied: their CV ROC-AUC differs by only 0.0014, which is within fold-to-fold noise, and their test ROC-AUC is identical. Round 1 was chosen because its learning rate is more conservative (0.1 vs 0.2). Tuning improved ROC-AUC by about 0.005 over the default, and further tuning plateaued, suggesting the pre-launch features limit performance rather than the hyperparameters.

**Limitation:** the model detects only about 59% of failed campaigns, because it leans toward predicting success.

In [16]:
from lightgbm import LGBMClassifier
import lightgbm

print(lightgbm.__version__)

lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(random_state=42, verbose=-1))
])
lgbm_pipeline.fit(X_train, y_train)

y_pred_lgbm = lgbm_pipeline.predict(X_test)
y_prob_lgbm = lgbm_pipeline.predict_proba(X_test)[:, 1]

lgbm_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_lgbm),
    "precision": precision_score(y_test, y_pred_lgbm),
    "recall": recall_score(y_test, y_pred_lgbm),
    "f1_score": f1_score(y_test, y_pred_lgbm),
    "roc_auc": roc_auc_score(y_test, y_prob_lgbm),
    "confusion_matrix": confusion_matrix(y_test, y_pred_lgbm)
}

print(f"Accuracy:  {lgbm_metrics['accuracy']:.3f}")
print(f"Precision: {lgbm_metrics['precision']:.3f}")
print(f"Recall:    {lgbm_metrics['recall']:.3f}")
print(f"F1-Score:  {lgbm_metrics['f1_score']:.3f}")
print(f"ROC AUC:   {lgbm_metrics['roc_auc']:.3f}")
print("Confusion Matrix:\n", lgbm_metrics["confusion_matrix"])

4.7.0
Accuracy:  0.698
Precision: 0.698
Recall:    0.793
F1-Score:  0.742
ROC AUC:   0.763
Confusion Matrix:
 [[2038 1448]
 [ 874 3341]]


In [17]:
lgbm_param_grid = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [15, 31, 63]
}

lgbm_search = GridSearchCV(lgbm_pipeline, lgbm_param_grid, scoring="roc_auc", cv=5, n_jobs=-1)
lgbm_search.fit(X_train, y_train)

print(lgbm_search.best_params_)
print(lgbm_search.best_score_)

best_lgbm = lgbm_search.best_estimator_
y_pred_lgbm_t = best_lgbm.predict(X_test)
y_prob_lgbm_t = best_lgbm.predict_proba(X_test)[:, 1]

lgbm_tuned_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_lgbm_t),
    "precision": precision_score(y_test, y_pred_lgbm_t),
    "recall": recall_score(y_test, y_pred_lgbm_t),
    "f1_score": f1_score(y_test, y_pred_lgbm_t),
    "roc_auc": roc_auc_score(y_test, y_prob_lgbm_t),
    "confusion_matrix": confusion_matrix(y_test, y_pred_lgbm_t)
}

print(f"Accuracy:  {lgbm_tuned_metrics['accuracy']:.3f}")
print(f"Precision: {lgbm_tuned_metrics['precision']:.3f}")
print(f"Recall:    {lgbm_tuned_metrics['recall']:.3f}")
print(f"F1-Score:  {lgbm_tuned_metrics['f1_score']:.3f}")
print(f"ROC AUC:   {lgbm_tuned_metrics['roc_auc']:.3f}")
print("Confusion Matrix:\n", lgbm_tuned_metrics["confusion_matrix"])

{'model__learning_rate': 0.05, 'model__n_estimators': 400, 'model__num_leaves': 15}
0.7619442506230013
Accuracy:  0.702
Precision: 0.701
Recall:    0.792
F1-Score:  0.744
ROC AUC:   0.765
Confusion Matrix:
 [[2066 1420]
 [ 878 3337]]


## XGBoost vs. LightGBM: Summary

Both gradient-boosted models were built on the same preprocessing pipeline and the same train/test split. Hyperparameters were tuned with 5-fold cross-validation on the training data only (scoring = ROC-AUC), and each final model was evaluated on the test set once.

| Model | CV ROC-AUC (train) | Accuracy | Precision | Recall | F1 | Test ROC-AUC |
|---|---|---|---|---|---|---|
| XGBoost (default) | n/a | 0.702 | 0.706 | 0.780 | 0.741 | 0.762 |
| XGBoost (tuned) | 0.7624 | 0.703 | 0.702 | 0.795 | 0.746 | 0.767 |
| LightGBM (default) | n/a | 0.698 | 0.698 | 0.793 | 0.742 | 0.763 |
| LightGBM (tuned) | 0.7619 | 0.702 | 0.701 | 0.792 | 0.744 | 0.765 |

**Best hyperparameters**
- XGBoost: `learning_rate=0.1`, `max_depth=3`, `n_estimators=400`
- LightGBM: `learning_rate=0.05`, `num_leaves=15`, `n_estimators=400`

(Library versions: XGBoost 3.4.1, LightGBM 4.7.0.)

### Key takeaways
- **The two libraries perform equivalently.** Test ROC-AUC differs by only 0.002 (0.767 vs. 0.765) and CV ROC-AUC by 0.0005, both within noise. This is expected, since both use the same boosting approach on the same features.
- **Tuning helped only slightly.** ROC-AUC improved by about 0.005 for XGBoost and 0.002 for LightGBM. A second, wider XGBoost search gave the same test ROC-AUC (0.767), so performance has plateaued.
- **The best models were the simplest ones.** Both searches favored shallow trees (depth 2-3, 15 leaves) and lower learning rates, which suggests the signal in the pre-launch features is mostly simple and additive.
- **The features, not the algorithm, are the limit.** Post-launch columns (updates, comments, backers, pledged amount) were excluded to avoid data leakage, so performance is capped near 0.76-0.77 ROC-AUC and about 70% accuracy.
- **Weakness: failed campaigns.** Both models lean toward predicting "success." They correctly identify only about 59% of failed campaigns (e.g., 2,066 of 3,486 for tuned LightGBM) while catching about 79% of successful ones.

### Selection
Of the two boosted models, **XGBoost (tuned)** is carried forward to the final model comparison (Task #8) because it has the highest CV and test ROC-AUC, although the margin over LightGBM is within noise.

In [18]:
import pandas as pd

results = pd.DataFrame([
    {"model": "XGBoost (default)", **{k: v for k, v in metrics.items() if k != "confusion_matrix"}},
    {"model": "XGBoost (tuned)", **{k: v for k, v in tuned_metrics.items() if k != "confusion_matrix"}},
    {"model": "LightGBM (default)", **{k: v for k, v in lgbm_metrics.items() if k != "confusion_matrix"}},
    {"model": "LightGBM (tuned)", **{k: v for k, v in lgbm_tuned_metrics.items() if k != "confusion_matrix"}},
]).round(3)

results
results.to_csv("../data/boosting_results.csv", index=False)